# Lab 04 — Image Restoration & Deconvolution

**Course:** Image Analysis and Processing — École des Mines de Saint-Étienne  
**Topics covered:**
1. Linear degradation modeling (motion blur PSF + additive Gaussian noise)
2. Direct Inverse Filtering and noise amplification analysis
3. Wiener Filtering in the frequency domain
4. Iterative Deconvolution: Van Cittert (VCA) and Richardson-Lucy (RLA)
5. Unsupervised Wiener deconvolution
6. Real-world astronomical deconvolution on **Hubble Space Telescope** FITS data (Jupiter & Saturn)

In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal
from scipy.ndimage import gaussian_filter
from astropy.io import fits
from skimage import restoration

LABS_DIR = Path(".")
FITS_DIR = Path("../Images/fits_data/fits_data")
rng = np.random.default_rng(42)

## 1. Helper Functions

We define utilities for generating a synthetic checkerboard target, adding controlled Gaussian noise, and converting a spatial Point Spread Function (PSF) into its centered Optical Transfer Function (OTF).

In [ ]:
def checkerboard(s: int = 8) -> np.ndarray:
    """Generate an 8x8 checkerboard with square size `s` pixels."""
    return np.kron([[1, 0] * 4, [0, 1] * 4] * 4, np.ones((s, s)))


def add_gaussian_noise(image: np.ndarray, sigma: float = 1000.0):
    """Add Gaussian noise to an image; higher `sigma` yields higher SNR."""
    img_min, img_max = np.min(image), np.max(image)
    noise = (img_max - img_min) / sigma * rng.standard_normal(image.shape)
    noisy = np.clip(image + noise, img_min, img_max)
    return noisy, image - noisy


def psf2otf(psf: np.ndarray, out_shape) -> np.ndarray:
    """Compute the Optical Transfer Function (OTF) from a PSF via centered FFT."""
    sh = np.array(psf.shape)
    s = np.array(out_shape)
    pad = s - sh
    h = np.pad(psf, ((0, pad[0]), (0, pad[1])), mode="constant")
    shift = (int(pad[0] / 2 + 1), int(pad[1] / 2 + 1))
    h_centered = np.fft.fftshift(np.roll(h, shift, axis=(0, 1)))
    return np.real(np.fft.fft2(h_centered, s))

## 2. Degradation Modeling

We convolve the checkerboard with a motion-blur PSF using circular boundary conditions (matching the periodicity assumption of the 2D FFT) and add Gaussian noise.

In [ ]:
C = checkerboard(s=8)
psf = np.load(LABS_DIR / "psf_motion.npy")

Cb = signal.convolve2d(C, psf, boundary="wrap", mode="same")
Cbn, noise = add_gaussian_noise(Cb, sigma=1000)

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, img, title in zip(
    axes,
    [C, psf, Cb, Cbn],
    ["Original", "Motion PSF", "Blurred", "Blurred + Noise"],
):
    ax.imshow(img, cmap="gray")
    ax.set_title(title)
    ax.axis("off")
plt.tight_layout()
plt.show()

## 3. Direct Inverse Filter

In the absence of noise, dividing by the OTF $H(u, v) + \alpha$ recovers the original image. When noise is present, small values of $H(u, v)$ catastrophically amplify high-frequency noise.

In [ ]:
H = psf2otf(psf, C.shape)
alpha = 1e-3

fr_clean = np.real(np.fft.ifft2(np.fft.fft2(Cb) / (H + alpha)))
fr_noisy = np.real(np.fft.ifft2(np.fft.fft2(Cbn) / (H + alpha)))

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, img, title in zip(
    axes,
    [C, fr_clean, fr_noisy],
    ["Original", "Inverse Filter (Noiseless)", "Inverse Filter (Noisy — Amplifies Noise)"],
):
    ax.imshow(img, cmap="gray")
    ax.set_title(title)
    ax.axis("off")
plt.tight_layout()
plt.show()

## 4. Wiener Filter

The Wiener filter regularizes the inversion using the noise-to-signal power spectral ratio $R = S_n / S_f$:

$$H_W(u, v) = \frac{H^*(u, v)}{|H(u, v)|^2 + R}$$

In [ ]:
# Noiseless Wiener (reduces to pseudo-inverse)
Hw_clean = np.where(np.abs(H) > 1e-12, 1.0 / np.where(np.abs(H) > 1e-12, H, 1.0), 0.0)
fr_wiener_clean = np.real(np.fft.ifft2(Hw_clean * np.fft.fft2(Cb)))

# Noisy Wiener using estimated noise-to-signal power ratio
Sn = np.abs(np.fft.fft2(noise)) ** 2
Sf = np.abs(np.fft.fft2(C)) ** 2
ratio = np.mean(Sn) / np.mean(Sf)

Hw = np.conj(H) / (np.abs(H) ** 2 + ratio)
fr_wiener = np.real(np.fft.ifft2(Hw * np.fft.fft2(Cbn)))

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, img, title in zip(
    axes,
    [C, fr_wiener_clean, fr_wiener],
    ["Original", "Wiener (Noiseless)", f"Wiener (Noisy, R={ratio:.2e})"],
):
    ax.imshow(img, cmap="gray")
    ax.set_title(title)
    ax.axis("off")
plt.tight_layout()
plt.show()

## 5. Iterative Deconvolution (Van Cittert & Richardson-Lucy)

Iterative methods avoid direct frequency inversion and naturally allow non-negativity constraints:
- **Van Cittert (VCA):** $f_{k+1} = f_k + \beta (g - h * f_k)$
- **Richardson-Lucy (RLA):** $f_{k+1} = f_k \cdot \left[ h(-x) * \frac{g}{h * f_k} \right]$

In [ ]:
def van_cittert(g: np.ndarray, psf: np.ndarray, n_iter: int = 200, beta: float = 0.01) -> np.ndarray:
    """Van Cittert iterative deconvolution."""
    H = psf2otf(psf, g.shape)
    fr = g.copy().astype(np.float64)
    for _ in range(n_iter):
        estimation = np.real(np.fft.ifft2(H * np.fft.fft2(fr)))
        fr = fr + beta * (g - estimation)
    return fr


def richardson_lucy(g: np.ndarray, psf: np.ndarray, n_iter: int = 10) -> np.ndarray:
    """Richardson-Lucy iterative deconvolution with non-negativity constraint."""
    H = psf2otf(psf, g.shape)
    fr = np.clip(g.copy().astype(np.float64), 1e-12, None)
    for _ in range(n_iter):
        estimation = np.real(np.fft.ifft2(H * np.fft.fft2(fr)))
        ratio = g / np.where(np.abs(estimation) > 1e-12, estimation, 1e-12)
        correction = np.real(np.fft.ifft2(np.conj(H) * np.fft.fft2(ratio)))
        fr = np.clip(fr * correction, 0.0, None)
    return fr


fr_vca = van_cittert(Cbn, psf, n_iter=200, beta=0.01)
fr_rla = richardson_lucy(Cbn, psf, n_iter=10)

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, img, title in zip(
    axes,
    [C, Cbn, fr_vca, fr_rla],
    ["Original", "Blurred + Noise", "Van Cittert (200 iter)", "Richardson-Lucy (10 iter)"],
):
    ax.imshow(img, cmap="gray")
    ax.set_title(title)
    ax.axis("off")
plt.tight_layout()
plt.show()

## 6. Unsupervised Wiener Deconvolution

Using `skimage.restoration.unsupervised_wiener`, the regularization hyperparameters are estimated automatically via Bayesian Gibbs sampling without requiring manual noise-to-signal ratio tuning.

In [ ]:
psf_gauss = np.zeros((15, 15))
psf_gauss[7, 7] = 1.0
psf_gauss = gaussian_filter(psf_gauss, sigma=2.0)

Cb_gauss = signal.convolve2d(C, psf_gauss, boundary="wrap", mode="same")
Cbn_gauss, _ = add_gaussian_noise(Cb_gauss, sigma=500)

fr_unsupervised, _ = restoration.unsupervised_wiener(Cbn_gauss, psf_gauss)

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, img, title in zip(
    axes,
    [C, psf_gauss, Cbn_gauss, fr_unsupervised],
    ["Original", "Gaussian PSF (sigma=2)", "Blurred + Noise", "Unsupervised Wiener"],
):
    ax.imshow(img, cmap="gray")
    ax.set_title(title)
    ax.axis("off")
plt.tight_layout()
plt.show()

## 7. Astronomical Deconvolution (Hubble Space Telescope)

We apply all four restoration methods to real FITS observations of **Jupiter** and **Saturn** captured by the Hubble Space Telescope using their corresponding instrumental PSFs.

In [ ]:
def load_fits(path: Path) -> np.ndarray:
    with fits.open(path, ignore_missing_end=True) as hdu:
        img = hdu[0].data if hdu[0].data is not None else hdu[1].data
        return np.nan_to_num(img.astype(np.float64))


# Load Jupiter (413 nm) & Saturn Hubble Space Telescope observations with their PSFs
jupiter = load_fits(FITS_DIR / "j413.fit")
psf_jup = load_fits(FITS_DIR / "psf_413.fit")
psf_jup /= psf_jup.sum()

saturn = load_fits(FITS_DIR / "saturn.fits")
psf_sat = load_fits(FITS_DIR / "saturn_psf.fits")
psf_sat /= psf_sat.sum()

fig, axes = plt.subplots(2, 2, figsize=(10, 9))
for ax, img, title in zip(
    axes.flat,
    [jupiter, psf_jup, saturn, psf_sat],
    ["Jupiter (HST)", "Jupiter PSF", "Saturn (HST)", "Saturn PSF"],
):
    ax.imshow(img, cmap="gray", origin="lower")
    ax.set_title(title)
    ax.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
def deconvolve_all(img: np.ndarray, psf: np.ndarray, n_iter: int = 10):
    """Run Inverse, Wiener, Van Cittert, and Richardson-Lucy deconvolution."""
    H = psf2otf(psf, img.shape)
    G = np.fft.fft2(img)

    fr_inv = np.real(np.fft.ifft2(G / (H + 1e-3)))

    R = 1e-3
    Hw = np.conj(H) / (np.abs(H) ** 2 + R)
    fr_wiener = np.real(np.fft.ifft2(Hw * G))

    fr_vca = van_cittert(img, psf, n_iter=n_iter, beta=1.0)
    fr_rla = richardson_lucy(img, psf, n_iter=n_iter)
    return fr_inv, fr_wiener, fr_vca, fr_rla


for name, obs, psf_obs in [("Jupiter", jupiter, psf_jup), ("Saturn", saturn, psf_sat)]:
    results = deconvolve_all(obs, psf_obs, n_iter=10)
    fig, axes = plt.subplots(1, 5, figsize=(20, 4))
    for ax, img, title in zip(
        axes,
        [obs, *results],
        [f"{name} (Original)", "Inverse Filter", "Wiener", "Van Cittert", "Richardson-Lucy"],
    ):
        ax.imshow(img, cmap="gray", origin="lower")
        ax.set_title(title)
        ax.axis("off")
    fig.suptitle(f"{name} Deconvolution Comparison (HST)", fontsize=13, fontweight="bold")
    plt.tight_layout()
    plt.show()